In [ ]:
# Step 1: Install MESA with adapters extra (D34)
!pip install "mesa-memory[adapters] @ git+https://github.com/Yasou13/MESA"

In [ ]:
# Step 2: Install Ollama and start background service
!apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import urllib.request

ollama_process = subprocess.Popen(["ollama", "serve"])

# Wait until Ollama answers
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434") as resp:
            if resp.status == 200:
                print("Ollama is ready!")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start within 60 seconds")

In [ ]:
# Step 3: Configure and pull internal models
LLM = "qwen2.5:7b"
EMBED = "bge-m3"

!ollama pull {LLM}
!ollama pull {EMBED}

In [ ]:
# Step 4: Write configuration to ~/.mesa/.env
from pathlib import Path

mesa_dir = Path.home() / ".mesa"
mesa_dir.mkdir(parents=True, exist_ok=True)
env_file = mesa_dir / ".env"
env_file.write_text(
    "MESA_MODEL_ENABLED=true\n"
    f"MESA_LLM_MODEL=ollama:{LLM}\n"
    f"MESA_EMBEDDING_MODEL=ollama:{EMBED}\n",
    encoding="utf-8",
)
print(f"Wrote {env_file}:\n{env_file.read_text()}")

In [ ]:
# Step 5: Start MESA server and wait for readiness
import json
import subprocess
import time

log_file = open("mesa.log", "w", encoding="utf-8")
mesa_proc = subprocess.Popen(["mesa", "start"], stdout=log_file, stderr=subprocess.STDOUT)

start_time = time.time()
ready = False
while time.time() - start_time < 300:
    res = subprocess.run(["mesa", "status", "--json"], capture_output=True, text=True)
    if res.returncode == 0:
        try:
            status = json.loads(res.stdout)
            if status.get("init", {}).get("status") in {"healthy", "ready"}:
                ready = True
                print("MESA is ready!")
                break
        except Exception:
            pass
    time.sleep(2)

if not ready:
    print("MESA failed to become ready within 5 minutes. Last 50 lines of mesa.log:")
    with open("mesa.log", "r", encoding="utf-8") as f:
        print("".join(f.readlines()[-50:]))
    raise RuntimeError("MESA server startup timed out")

In [ ]:
# Step 6: Add document, search via CLI, and retrieve context via SDK
sample_text = (
    "Madde 1- Kiracı, kira bedelini her ayın beşinci gününe kadar ev sahibinin "
    "banka hesabına ödemekle yükümlüdür. Zamanında ödenmeyen kira bedeli için "
    "yasal faiz uygulanır."
)
with open("kira_sozlesmesi.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

# Add document via CLI
!mesa add kira_sozlesmesi.txt

# Search via CLI
!mesa search "kira ne zaman ödenir?"

# Query context using Python SDK
from mesa_client import Mesa

mesa = Mesa()
context = mesa.context("kira ödeme tarihi")
print("\nRetrieved Evidence Context:\n")
print(context)

### Alternative: vLLM instead of Ollama

To use vLLM on Colab instead of Ollama:
Start vLLM on port 8000 (serving an OpenAI-compatible API), and start MESA on another port (`MESA_PORT=8001`), allowed by D31 on local loopback.

In `~/.mesa/.env`:

```bash
MESA_PORT=8001
MESA_MODEL_ENABLED=true
MESA_LLM_MODEL=openai_compatible:qwen2.5-7b
MESA_LLM_BASE_URL=http://127.0.0.1:8000/v1
```
